In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', None)

In [2]:
icd9 = pd.read_json('../data/icd9-cm-2015-code.json').T
icd9 = icd9.reset_index(names="Code")
icd9

/tmp/ipykernel_2067464/641046097.py:1: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  icd9 = pd.read_json('../data/icd9-cm-2015-code.json').T
/tmp/ipykernel_2067464/641046097.py:1: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  icd9 = pd.read_json('../data/icd9-cm-2015-code.json').T
/tmp/ipykernel_2067464/641046097.py:1: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matchin

,Code,code,title,exclude,sex,synonym,include,codeAlso,note,fourth-digit
0,00,00,"Procedures and interventions , Not Elsewhere Classified",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,000,00.0,Therapeutic ultrasound,"[diagnostic ultrasound (non-invasive) (88.71-88.79), intracardiac echocardiography [ICE] (heart chamber(s)) (37.28), intravascular imaging (adjunctive) (00.21-00.29)]",NaN,NaN,NaN,NaN,NaN,NaN
2,0001,00.01,Therapeutic ultrasound of vessels of head and neck,"[diagnostic ultrasound of: eye (95.13), diagnostic ultrasound of: head and neck (88.71), Therapeutic ultrasound of vessels of head and neck that of inner ear (20.79), ultrasonic: angioplasty of non-coronary vessel (39.50), ultrasonic: embolectomy (38.01, 38.02), ultrasonic: endarterectomy (38.11, 38.12), ultrasonic: thrombectomy (38.01, 38.02), diagnostic ultrasound (non-invasive) (88.71-88.79), intracardiac echocardiography [ICE] (heart chamber(s)) (37.28), intravascular imaging (adjunctive) (00.21-00.29)]",B,"[Anti-restenotic ultrasound, Intravascular non-ablative ultrasound]",NaN,NaN,NaN,NaN
3,0002,00.02,Therapeutic ultrasound of heart,"[diagnostic ultrasound of heart (88.72), ultrasonic ablation of heart lesion (37.34), ultrasonic angioplasty of coronary vessels (00.66, 36.09), diagnostic ultrasound (non-invasive) (88.71-88.79), intracardiac echocardiography [ICE] (heart chamber(s)) (37.28), intravascular imaging (adjunctive) (00.21-00.29)]",B,"[Anti-restenotic ultrasound, Intravascular non-ablative ultrasound]",NaN,NaN,NaN,NaN
4,0003,00.03,Therapeutic ultrasound of peripheral vascular vessels,"[diagnostic ultrasound of peripheral vascular system (88.77), ultrasonic angioplasty of: non-coronary vessel (39.50), diagnostic ultrasound (non-invasive) (88.71-88.79), intracardiac echocardiography [ICE] (heart chamber(s)) (37.28), intravascular imaging (adjunctive) (00.21-00.29)]",B,"[Anti-restenotic ultrasound, Intravascular non-ablative ultrasound]",NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
4641,9995,99.95,Stretching of foreskin,NaN,B,NaN,NaN,NaN,NaN,NaN
4642,9996,99.96,Collection of sperm for artificial insemination,NaN,M,NaN,NaN,NaN,NaN,NaN
4643,9997,99.97,Fitting of denture,NaN,B,NaN,NaN,NaN,NaN,NaN
4644,9998,99.98,Extraction of milk from lactating breast,NaN,F,NaN,NaN,NaN,NaN,NaN


In [3]:
icd9_codeAlso = icd9[['Code', 'codeAlso']].dropna()
icd9_codeAlso = icd9_codeAlso.explode('codeAlso')
icd9_codeAlso["codeAlso"] = icd9_codeAlso['codeAlso'].str.extract(r'.*\(([^)]+)\)[^()]*$')
icd9_codeAlso = icd9_codeAlso.dropna()
icd9_codeAlso["codeAlso"] = icd9_codeAlso["codeAlso"].str.replace('.','')
icd9_codeAlso

,Code,codeAlso
32,004,"0061- 0062, 0066, 3950"
32,004,1753-1756
32,004,3810 3818
32,004,"0055, 0063-0065, 3606 -3607, 3990"
32,004,3609
...,...,...
4571,992,"4695, 4696"
4571,992,5595
4571,992,5093
4571,992,3996


In [4]:
def expand_icd9(interval_string : str, valid_codes : list[str]):
    """
    Expands ICD-9 strings into individual codes, 
    filtering against a list of known valid codes.
    """
    valid_set = set(valid_codes)
    expanded_codes = []

    parts = [p.strip() for p in interval_string.split(',')]

    for part in parts:
        if '-' in part:
            # Step 1: Split the range (e.g., "0751-0759")
            start, end = part.split('-')
            start = start.strip()
            end = end.strip()

            # Step 3: Track the length to maintain leading zeros (e.g., 4 characters)
            num_length = len(start)
            if len(start) < len(end):
                start = start + '0'

            # Step 4: Generate the sequence
            for num in range(int(start), int(end) + 1):
                generated_code = str(num).zfill(num_length)
                
                # Step 5: Only add it if it actually exists in your data
                if generated_code in valid_set:
                    expanded_codes.append(generated_code)

        else:
            base_code = part.strip()
            
            # Loop through the valid set to find the exact code AND any detailed sub-codes
            for valid_code in valid_set:
                if valid_code.startswith(base_code):
                    expanded_codes.append(valid_code)

    return sorted(expanded_codes)

In [5]:
icd9_codeAlso['codeAlso'] = icd9_codeAlso['codeAlso'].str.replace(r'(\d+)\s+(\d+)', r'\1,\2', regex=True)
icd9_codeAlso["codeAlso_expand"] = icd9_codeAlso['codeAlso'].apply(
    lambda x: sorted(list(set(
        expanded_code 
        for c in x.split(',')
        for expanded_code in expand_icd9(c, icd9['Code'].tolist())
    )))
)
icd9_codeAlso = icd9_codeAlso[icd9_codeAlso['codeAlso_expand'].apply(len) != 0]
icd9_codeAlso

,Code,codeAlso,codeAlso_expand
32,004,"0061- 0062, 0066, 3950","[0061, 0062, 0066, 3950]"
32,004,1753-1756,"[1753, 1754, 1755, 1756]"
32,004,"3810,3818","[3810, 3818]"
32,004,"0055, 0063-0065, 3606 -3607, 3990","[0055, 0063, 0064, 0065, 3606, 3607, 3990]"
32,004,3609,[3609]
...,...,...,...
4571,992,"4695, 4696","[4695, 4696]"
4571,992,5595,[5595]
4571,992,5093,[5093]
4571,992,3996,[3996]


In [6]:
icd9_codeAlso.sample(10)

,Code,codeAlso,codeAlso_expand
2659,684,6531-6564,"[6531, 6539, 6541, 6549, 6551, 6552, 6553, 6554, 6561, 6562, 6563, 6564]"
192,0390,8606,[8606]
2662,685,7079,[7079]
1414,3814,0045-0048,"[0045, 0046, 0047, 0048]"
4571,992,3997,[3997]
738,1751,"3610,3619","[3610, 3619]"
1549,3990,0045-0048,"[0045, 0046, 0047, 0048]"
1302,3609,0044,[0044]
1511,3950,0040-0043,"[0040, 0041, 0042, 0043]"
3055,7839,7810-7819,"[7810, 7811, 7812, 7813, 7814, 7815, 7816, 7817, 7818, 7819]"


In [7]:
icd9_codeAlso.query('Code == "1756"')

,Code,codeAlso,codeAlso_expand
743,1756,9910,[9910]
743,1756,0055,[0055]
743,1756,3990,[3990]
743,1756,0045-0048,"[0045, 0046, 0047, 0048]"
743,1756,0040-0043,"[0040, 0041, 0042, 0043]"
743,1756,0044,[0044]


In [8]:
final_table = icd9_codeAlso.explode('codeAlso_expand').groupby('Code').agg(list)
final_table = final_table.drop(columns = 'codeAlso')
final_table.columns = ['codeAlso']
final_table_dict = {i: row['codeAlso'] for i,row in final_table.iterrows()}
final_table_dict

{'004': ['0061',
  '0062',
  '0066',
  '3950',
  '1753',
  '1754',
  '1755',
  '1756',
  '3810',
  '3818',
  '0055',
  '0063',
  '0064',
  '0065',
  '3606',
  '3607',
  '3990',
  '3609'],
 '0049': ['9910',
  '3606',
  '3607',
  '3604',
  '0045',
  '0046',
  '0047',
  '0048',
  '0040',
  '0041',
  '0042',
  '0043',
  '3603',
  '3609',
  '0066',
  '0044',
  '1755'],
 '0055': ['3950', '1756', '0045', '0048', '0040', '0043', '0044'],
 '0056': ['0057'],
 '0057': ['0056'],
 '0060': ['3950',
  '1756',
  '3990',
  '0045',
  '0046',
  '0047',
  '0048',
  '0040',
  '0041',
  '0042',
  '0043',
  '0044'],
 '0061': ['9910',
  '1753',
  '0063',
  '0064',
  '0045',
  '0048',
  '0040',
  '0043',
  '0044'],
 '0062': ['9910', '1754', '0065', '0045', '0048', '0040', '0043', '0044'],
 '0063': ['0045', '0048', '0040', '0043', '0061', '1753', '0044'],
 '0064': ['0045', '0048', '0040', '0043', '0061', '1753', '0044'],
 '0065': ['0045', '0048', '0040', '0043', '0062', '1754', '0044'],
 '0066': ['9910',
  '360

In [9]:
import json
with open('../result/icd9_codeAlso.json', 'w') as f:
    json.dump(final_table_dict, f, indent=4)